In [ ]:


!pip install -q google-genai pandas numpy scikit-learn pypdf gradio ipywidgets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = Path("/content/drive/MyDrive/MIE1624_chatbot_project")

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
KNOWLEDGE_DIR = DATA_DIR / "knowledge"
APP_DIR = PROJECT_ROOT / "app"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
LOG_DIR = OUTPUTS_DIR / "logs"
EXPORT_DIR = OUTPUTS_DIR / "exports"

for p in [RAW_DIR, PROCESSED_DIR, KNOWLEDGE_DIR, APP_DIR, OUTPUTS_DIR, LOG_DIR, EXPORT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_DIR exists:", RAW_DIR.exists())
print("KNOWLEDGE_DIR exists:", KNOWLEDGE_DIR.exists())

PROJECT_ROOT: /content/drive/MyDrive/MIE1624_chatbot_project
RAW_DIR exists: True
KNOWLEDGE_DIR exists: True


In [ ]:
import os
from getpass import getpass

if "GEMINI_API_KEY" not in os.environ:
    os.environ["GEMINI_API_KEY"] = getpass("Paste your Gemini API key: ")

MODEL_NAME = os.getenv("GEMINI_MODEL", "gemini-3.1-pro-preview")

Paste your Gemini API key: ··········


In [ ]:
import re
import json
import math
import textwrap
from typing import List, Dict, Any, Optional

import numpy as np
import pandas as pd
from pypdf import PdfReader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from google import genai

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

In [ ]:
def read_text_file(path: Path) -> str:
    try:
        return path.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        return path.read_text(encoding="latin-1")

def read_pdf_file(path: Path) -> str:
    reader = PdfReader(str(path))
    pages = []
    for page in reader.pages:
        pages.append(page.extract_text() or "")
    return "\n".join(pages)

def load_any_text(path: Path) -> str:
    suffix = path.suffix.lower()
    if suffix in [".md", ".txt"]:
        return read_text_file(path)
    if suffix == ".pdf":
        return read_pdf_file(path)
    return ""

def safe_read_csv(path: Path) -> Optional[pd.DataFrame]:
    if not path.exists():
        return None
    try:
        return pd.read_csv(path)
    except Exception as e:
        print(f"Failed to read {path.name}: {e}")
        return None

In [ ]:
DOCS_PATH = RAW_DIR / "ai_sentiment_all_documents.csv"
DB_PATH = RAW_DIR / "db_merged.csv"

docs_df = safe_read_csv(DOCS_PATH)
db_merged_df = safe_read_csv(DB_PATH)

if docs_df is None:
    raise FileNotFoundError(f"Missing required file: {DOCS_PATH}")

if db_merged_df is None:
    raise FileNotFoundError(f"Missing required file: {DB_PATH}")

print("docs_df shape:", docs_df.shape)
print("db_merged_df shape:", db_merged_df.shape)
display(docs_df.head(2))
display(db_merged_df.head(2))

docs_df shape: (590, 8)
db_merged_df shape: (40, 32)


,country,source_type,title,text,date,url,sentiment_score,sentiment_label
0,Argentina,NewsAPI,PromptSpy Android Malware Abuses Gemini AI to ...,PromptSpy Android Malware Abuses Gemini AI to ...,2026-02-19T17:52:00Z,https://thehackernews.com/2026/02/promptspy-an...,-0.6808,Negative
1,Argentina,NewsAPI,Android malware taps Gemini to navigate infect...,Android malware taps Gemini to navigate infect...,2026-02-19T16:04:52Z,https://www.theregister.com/2026/02/19/genai_m...,0.5106,Positive


,Country,Average Gross,Average Gross (%GDP),publications,citations,Private Investment ($B) (2013-2024),Newly Founded AI Companies (2013-2024),Ai and ML(Popularity),AI_usage_score,AI_usage_percent,...,pct_patents_accepted,accepted_patent_per_capita,Rank,Working_Age_Pop_Using_AI,Median_Broadband _download_speed_(MBPS),Mean_Download_speed_(MBPS),Number_of_SuperComputers,nb_of_computers_in_Top500,Max_performance(TFlops),Computing_power_per_capita
0,US,766897.6930,3.392855,15.386694,27.070086,471,"6,956",18.0,8.295607,48.797686,...,78.978804,0.000873,1,28.3,302.68,161.97,171,172,6475559,0.019429
1,CN,665572.5757,2.402397,21.245047,19.197077,119,"1,605",72.0,11.540426,67.884856,...,45.976365,0.000727,2,16.3,223.47,37.56,40,63,319062,0.000226


In [ ]:
docs_df = docs_df.copy()

for col in ["country", "source_type", "title", "text", "date", "url", "sentiment_score", "sentiment_label"]:
    if col not in docs_df.columns:
        docs_df[col] = ""

docs_df["country"] = docs_df["country"].fillna("").astype(str)
docs_df["source_type"] = docs_df["source_type"].fillna("").astype(str)
docs_df["title"] = docs_df["title"].fillna("").astype(str)
docs_df["text"] = docs_df["text"].fillna("").astype(str)
docs_df["date"] = docs_df["date"].fillna("").astype(str)
docs_df["url"] = docs_df["url"].fillna("").astype(str)

docs_df["sentiment_score"] = pd.to_numeric(docs_df["sentiment_score"], errors="coerce").fillna(0.0)

if docs_df["sentiment_label"].astype(str).str.strip().eq("").all():
    docs_df["sentiment_label"] = np.where(
        docs_df["sentiment_score"] > 0.05, "positive",
        np.where(docs_df["sentiment_score"] < -0.05, "negative", "neutral")
    )
else:
    docs_df["sentiment_label"] = docs_df["sentiment_label"].fillna("").astype(str).str.lower()

country_summary = (
    docs_df.groupby("country", dropna=False)
    .agg(
        n_documents=("country", "size"),
        avg_sentiment=("sentiment_score", "mean"),
        positive_share=("sentiment_label", lambda s: (s == "positive").mean()),
        neutral_share=("sentiment_label", lambda s: (s == "neutral").mean()),
        negative_share=("sentiment_label", lambda s: (s == "negative").mean()),
    )
    .reset_index()
    .sort_values("avg_sentiment", ascending=False)
)

source_country_summary = (
    docs_df.groupby(["country", "source_type"], dropna=False)
    .agg(
        n_documents=("country", "size"),
        avg_sentiment=("sentiment_score", "mean"),
        positive_share=("sentiment_label", lambda s: (s == "positive").mean()),
        negative_share=("sentiment_label", lambda s: (s == "negative").mean()),
    )
    .reset_index()
)

country_summary_path = PROCESSED_DIR / "ai_sentiment_country_summary.csv"
source_country_summary_path = PROCESSED_DIR / "ai_sentiment_source_country_summary.csv"

country_summary.to_csv(country_summary_path, index=False)
source_country_summary.to_csv(source_country_summary_path, index=False)

print("Saved:", country_summary_path.name, source_country_summary.shape)
print("Saved:", source_country_summary_path.name, source_country_summary.shape)

display(country_summary.head(5))
display(source_country_summary.head(5))

Saved: ai_sentiment_country_summary.csv (41, 6)
Saved: ai_sentiment_source_country_summary.csv (41, 6)


,country,n_documents,avg_sentiment,positive_share,neutral_share,negative_share
2,Austria,14,0.572371,0.857143,0.000000,0.142857
24,Norway,15,0.550340,0.866667,0.000000,0.133333
21,Lithuania,15,0.542107,0.800000,0.133333,0.066667
11,Finland,15,0.524080,0.866667,0.066667,0.066667
15,Hungary,13,0.482115,0.846154,0.000000,0.153846


,country,source_type,n_documents,avg_sentiment,positive_share,negative_share
0,Argentina,NewsAPI,15,0.073653,0.533333,0.266667
1,Australia,NewsAPI,15,0.110587,0.533333,0.400000
2,Austria,NewsAPI,14,0.572371,0.857143,0.142857
3,Belgium,NewsAPI,14,0.387629,0.857143,0.142857
4,Bulgaria,NewsAPI,13,-0.138454,0.461538,0.461538


In [ ]:
knowledge_files = sorted([
    p for p in KNOWLEDGE_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in [".md", ".pdf", ".txt"]
])

knowledge_docs = []

for path in knowledge_files:
    text = load_any_text(path).strip()
    if text:
        knowledge_docs.append({
            "source": path.name,
            "text": text
        })

print("Knowledge files loaded:")
for d in knowledge_docs:
    print("-", d["source"], f"({len(d['text'])} chars)")

Knowledge files loaded:
- AI-Adopt Tax Credit (Addressing Enterprise Usage).md (1230 chars)
- Conclusion.md (797 chars)
- Current Research and Existing Metrics.md (1469 chars)
- Data Collection  Cleaning Feature Selection & Justification.md (2441 chars)
- Identifying Top 10 Contributing Factors.md (688 chars)
- Josue Performance Metrics.pdf (9632 chars)
- LLM Chatbot.md (2591 chars)
- Part1-CurrentStateOfAI.md (8258 chars)
- Practical steps for implementing AI Strategy.md (4663 chars)
- Random Forest Model.md (4138 chars)
- The Mitacs AI-Bridge Internship (Addressing Talent & Industry).md (2655 chars)
- The Safe-Canada Certification & PR Campaign (Addressing Sentiment).md (1782 chars)
- Visualization of the Proposed AI Strategy.md (2848 chars)
- XGBoost Model.md (1512 chars)
- canada_recommendations.md (3182 chars)
- data_preprocessing.md (2288 chars)
- feature_definitions.md (3834 chars)
- final_report_master_summary.pdf (41640 chars)
- introduction.md (1077 chars)
- project_master_su

In [ ]:
def chunk_text(text: str, chunk_size: int = 1200, overlap: int = 150) -> List[str]:
    text = re.sub(r"\s+", " ", text).strip()
    if not text:
        return []
    chunks = []
    start = 0
    n = len(text)
    while start < n:
        end = min(start + chunk_size, n)
        chunks.append(text[start:end])
        if end == n:
            break
        start = max(end - overlap, 0)
    return chunks

knowledge_chunks = []
for doc in knowledge_docs:
    for i, chunk in enumerate(chunk_text(doc["text"])):
        knowledge_chunks.append({
            "source": doc["source"],
            "chunk_id": i,
            "text": chunk
        })

print("Knowledge chunks:", len(knowledge_chunks))

Knowledge chunks: 94


In [ ]:
article_docs = []
for _, row in docs_df.iterrows():
    article_text = (
        f"Country: {row['country']}. "
        f"Source: {row['source_type']}. "
        f"Title: {row['title']}. "
        f"Date: {row['date']}. "
        f"URL: {row['url']}. "
        f"Sentiment: {row['sentiment_label']} ({row['sentiment_score']:.3f}). "
        f"Text: {row['text']}"
    )
    article_docs.append({
        "source": f"article::{row['title'][:80]}",
        "country": row["country"],
        "text": article_text
    })

retrieval_corpus = []

for x in knowledge_chunks:
    retrieval_corpus.append({
        "kind": "knowledge",
        "source": x["source"],
        "country": "",
        "text": x["text"]
    })

for x in article_docs:
    retrieval_corpus.append({
        "kind": "article",
        "source": x["source"],
        "country": x["country"],
        "text": x["text"]
    })

retrieval_df = pd.DataFrame(retrieval_corpus)
retrieval_df["text"] = retrieval_df["text"].fillna("").astype(str)

vectorizer = TfidfVectorizer(stop_words="english", max_features=20000, ngram_range=(1, 2))
retrieval_matrix = vectorizer.fit_transform(retrieval_df["text"])

print("Retrieval corpus size:", len(retrieval_df))

Retrieval corpus size: 684


In [ ]:
COUNTRY_NAME_TO_CODE = {
    "argentina": "AR", "austria": "AT", "australia": "AU", "belgium": "BE",
    "bulgaria": "BG", "canada": "CA", "switzerland": "CH", "chile": "CL",
    "china": "CN", "colombia": "CO", "czech republic": "CZ", "czechia": "CZ",
    "germany": "DE", "denmark": "DK", "spain": "ES", "finland": "FI",
    "france": "FR", "united kingdom": "GB", "uk": "GB", "great britain": "GB",
    "greece": "GR", "hungary": "HU", "israel": "IL", "india": "IN",
    "italy": "IT", "japan": "JP", "south korea": "KR", "korea": "KR",
    "lithuania": "LT", "mexico": "MX", "netherlands": "NL", "norway": "NO",
    "new zealand": "NZ", "poland": "PL", "portugal": "PT", "romania": "RO",
    "sweden": "SE", "singapore": "SG", "slovenia": "SI", "slovakia": "SK",
    "turkey": "TR", "united states": "US", "usa": "US", "us": "US",
}

ALL_COUNTRIES = sorted([str(x) for x in docs_df["country"].dropna().unique() if str(x).strip()])

def normalize_text(s: str) -> str:
    return str(s).strip().lower()

def to_country_code(name_or_code: str) -> str:
    x = normalize_text(name_or_code)
    if len(x) == 2 and x.isalpha():
        return x.upper()
    return COUNTRY_NAME_TO_CODE.get(x, x.upper())

def detect_countries_from_query(query: str) -> List[str]:
    q = query.lower()
    found = [c for c in ALL_COUNTRIES if c.lower() in q]
    for k in COUNTRY_NAME_TO_CODE.keys():
        t = k.title()
        if k in q and t not in found:
            found.append(t)
    return sorted(set(found))

def get_country_summary(country_names: Optional[List[str]] = None) -> List[Dict[str, Any]]:
    df = country_summary.copy()
    if country_names:
        df = df[df["country"].isin(country_names)]
    return df.to_dict(orient="records")

def get_source_country_summary(country_names: Optional[List[str]] = None) -> List[Dict[str, Any]]:
    df = source_country_summary.copy()
    if country_names:
        df = df[df["country"].isin(country_names)]
    return df.to_dict(orient="records")

def get_db_merged_rows(country_names: Optional[List[str]] = None) -> List[Dict[str, Any]]:
    df = db_merged_df.copy()
    country_col = None
    for c in df.columns:
        if c.lower().strip() in {"country", "countries"}:
            country_col = c
            break
    if country_col is None:
        return []

    df["_country_code_norm"] = df[country_col].astype(str).str.strip().str.upper()
    if country_names:
        codes = [to_country_code(x) for x in country_names]
        df = df[df["_country_code_norm"].isin(codes)]
    return df.to_dict(orient="records")

In [ ]:
def retrieve_context(query: str, top_k: int = 6, country_filter: Optional[List[str]] = None) -> List[Dict[str, Any]]:
    work_df = retrieval_df.copy()

    if country_filter:
        mask = (work_df["country"].isin(country_filter)) | (work_df["kind"] == "knowledge")
        work_df = work_df[mask].copy()

    if work_df.empty:
        return []

    local_matrix = vectorizer.transform(work_df["text"])
    q_vec = vectorizer.transform([query])
    sims = cosine_similarity(q_vec, local_matrix).flatten()

    idxs = np.argsort(-sims)[:top_k]
    rows = work_df.iloc[idxs].copy()
    rows["score"] = sims[idxs]

    out = []
    for _, row in rows.iterrows():
        out.append({
            "kind": row["kind"],
            "source": row["source"],
            "country": row["country"],
            "score": float(row["score"]),
            "text": str(row["text"])[:1500]
        })
    return out

In [ ]:
XGB_FEATURES_PATH = KNOWLEDGE_DIR / "XGBoost Top 10 Features.csv"
xgb_features_df = safe_read_csv(XGB_FEATURES_PATH)

if xgb_features_df is not None:
    print("Loaded xgb_features_df:", xgb_features_df.shape)
    display(xgb_features_df.head())
else:
    print("XGBoost feature file not found.")

Loaded xgb_features_df: (10, 2)


,Features,Importance Score
0,accepted_patent_per_capita,0.310611
1,publications,0.153911
2,Average_Gross,0.124359
3,Median_Broadband__download_speed_MBPS,0.074728
4,total_data_centers_per_capita,0.062208


In [ ]:
xgb_feature_docs = []

if xgb_features_df is not None:
    for _, row in xgb_features_df.iterrows():
        feature = str(row.iloc[0])
        score = str(row.iloc[1])
        xgb_feature_docs.append({
            "kind": "knowledge",
            "source": "XGBoost Top 10 Features.csv",
            "country": "",
            "text": f"XGBoost feature importance. Feature: {feature}. Importance score: {score}."
        })

In [ ]:
for x in xgb_feature_docs:
    retrieval_corpus.append(x)
    retrieval_df = pd.DataFrame(retrieval_corpus)
retrieval_df["text"] = retrieval_df["text"].fillna("").astype(str)

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=20000,
    ngram_range=(1, 2)
)
retrieval_matrix = vectorizer.fit_transform(retrieval_df["text"])

print("Rebuilt retrieval corpus size:", len(retrieval_df))
print(retrieval_df.tail(12))

Rebuilt retrieval corpus size: 694
          kind                                             source  \
682    article  article::AI-focused DreamzTech Solutions named...   
683    article  article::South Africa Data Center Market Inves...   
684  knowledge                        XGBoost Top 10 Features.csv   
685  knowledge                        XGBoost Top 10 Features.csv   
686  knowledge                        XGBoost Top 10 Features.csv   
687  knowledge                        XGBoost Top 10 Features.csv   
688  knowledge                        XGBoost Top 10 Features.csv   
689  knowledge                        XGBoost Top 10 Features.csv   
690  knowledge                        XGBoost Top 10 Features.csv   
691  knowledge                        XGBoost Top 10 Features.csv   
692  knowledge                        XGBoost Top 10 Features.csv   
693  knowledge                        XGBoost Top 10 Features.csv   

          country                                               tex

In [ ]:
def call_llm(prompt: str) -> str:
    try:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt
        )
        return response.text
    except Exception as e:
        return f"LLM error: {str(e)}"

In [ ]:
def web_search_tool(query: str):
    return {
        "tool": "web_search_placeholder",
        "result": f"Simulated external search for query: {query}"
    }

In [ ]:
WRITER_SYSTEM = """
You are a business-oriented AI strategy assistant for Canada.

Rules:
1. Answer clearly for a non-technical audience.
2. Use the supplied evidence only, but give the strongest possible answer from the available evidence.
3. If the user asks for recommendations, prioritize recommendation-specific sources such as canada_recommendations.md before broader background files.
4. If retrieved evidence is used, mention sources like [Source 1], [Source 2].
5. If structured data is used, say which table it came from.
6. If evidence is partially incomplete, mention the limitation briefly near the end rather than leading with it, unless the missing evidence prevents a meaningful answer.
7. End with a short section called 'Data used'.
8. Keep answers focused on Canada's AI competitiveness, strategy, methodology, and recommendations.
9. Do not begin with broad disclaimers about incomplete evidence if relevant recommendation or structured evidence is available.
"""

def retrieve_context(query: str, top_k: int = 6, country_filter=None):
    work_df = retrieval_df.copy()

    if country_filter:
        mask = (work_df["country"].isin(country_filter)) | (work_df["kind"] == "knowledge")
        work_df = work_df[mask].copy()

    if work_df.empty:
        return []

    local_matrix = vectorizer.transform(work_df["text"])
    q_vec = vectorizer.transform([query])
    sims = cosine_similarity(q_vec, local_matrix).flatten()

    idxs = np.argsort(-sims)[:top_k]
    rows = work_df.iloc[idxs].copy()
    rows["score"] = sims[idxs]

    out = []
    for _, row in rows.iterrows():
        out.append({
            "kind": row["kind"],
            "source": row["source"],
            "country": row["country"],
            "score": float(row["score"]),
            "text": str(row["text"])[:1500]
        })
    return out


def planner_agent(user_query: str):
    q = user_query.lower()
    country_filters = detect_countries_from_query(user_query)

    needs_retrieval = any(k in q for k in [
    "why", "evidence", "article", "news", "explain", "strategy", "policy",
    "recommend", "trust", "sentiment", "adoption", "implementation",
    "xgboost", "xg boost", "feature", "importance",
    "measure", "measured", "method", "methodology", "competitiveness",
    "how do you measure", "how is ai competitiveness measured"
    ])

    needs_structured_data = any(k in q for k in [
    "rank", "top", "compare", "table", "value", "score", "metric",
    "how many", "show me", "row", "list", "sentiment",
    "xgboost", "xg boost", "feature", "importance",
    "measure", "measured", "method", "methodology", "competitiveness"
    ])

    needs_python_tool = any(k in q for k in [
        "rank", "top", "compare", "highest", "lowest", "best", "worst",
        "xgboost", "xg boost", "feature", "importance"
    ])
    needs_critique = any(k in q for k in [
    "critique", "critic", "weakness", "risk", "counterargument",
    "revise", "revision", "improve the recommendation", "self-correction"
    ])

    needs_web = any(k in q for k in [
        "latest", "most recent", "government announced", "search",
        "gap analysis", "last 2 years", "recent developments"
    ])

    needs_projection = any(k in q for k in [
        "project", "projection", "impact", "assumption", "projected",
        "cost too much", "relative to impact", "specific numbers"
    ])

    show_workflow = any(k in q for k in [
        "show each capability", "label which agent", "agent speaking",
        "demonstrate capabilities", "workflow"
    ])

    return {
    "needs_retrieval": needs_retrieval,
    "needs_structured_data": needs_structured_data,
    "needs_python_tool": needs_python_tool,
    "needs_critique": needs_critique,
    "needs_web": needs_web,
    "needs_projection": needs_projection,
    "show_workflow": show_workflow,
    "country_filters": country_filters,
    "reason": "Keyword-based planner v2"
}


def researcher_agent(user_query: str, country_filters=None):
    q = user_query.lower()

    is_recommendation_query = any(k in q for k in [
        "recommend", "recommendation", "policy", "improve canada"
    ])

    use_tool = any(k in q for k in [
        "latest", "recent", "news", "2025", "update"
    ])

    if is_recommendation_query:
        rec_hits = retrieval_df[
            retrieval_df["source"].str.contains("recommendations", case=False, na=False)
        ]

        normal_hits = retrieve_context(user_query, top_k=6, country_filter=country_filters)

        hits = []

        if not rec_hits.empty:
            rec_hits = rec_hits.head(3)
            for _, row in rec_hits.iterrows():
                hits.append({
                    "kind": row["kind"],
                    "source": row["source"],
                    "country": row["country"],
                    "score": 1.9,
                    "text": row["text"]
                })

        hits += normal_hits

    else:
        hits = retrieve_context(user_query, top_k=6, country_filter=country_filters)

    # dedup
    seen = set()
    dedup_hits = []
    for h in hits:
        key = (h["source"], h["text"][:100])
        if key not in seen:
            seen.add(key)
            dedup_hits.append(h)

    # context build
    blocks = []
    for i, hit in enumerate(dedup_hits[:6], 1):
        blocks.append(
            f"[Source {i}] file={hit['source']} | score={hit['score']:.4f}\n{hit['text']}"
        )

    # ⭐ NEW: tool usage
    tool_output = ""
    if use_tool:
        tool_output = web_search_tool(user_query)

    return {
        "research_context": "\n\n".join(blocks),
        "tool_context": tool_output,
        "hits": dedup_hits[:6]
    }

def analyst_agent(user_query: str, country_filters=None):
    if country_filters is None:
        country_filters = []

    # 先把结构化数据准备好
    country_summary_data = get_country_summary(country_filters)
    source_country_summary_data = get_source_country_summary(country_filters)
    db_merged_rows_data = get_db_merged_rows(country_filters)

    structured = {
        "country_summary": country_summary_data,
        "source_country_summary": source_country_summary_data,
        "db_merged_rows": db_merged_rows_data,
        "xgb_features": xgb_features_df.to_dict(orient="records")
        if ("xgb_features_df" in globals() and xgb_features_df is not None)
        else []
    }

    q = user_query.lower()
    tool_result = None

    try:
        # 转成 DataFrame，方便分析
        country_summary_df = pd.DataFrame(country_summary_data) if country_summary_data else pd.DataFrame()
        db_merged_df_local = pd.DataFrame(db_merged_rows_data) if db_merged_rows_data else pd.DataFrame()

        # 1) XGBoost top features
        if ("xgboost" in q or "xg boost" in q) and ("feature" in q or "importance" in q or "top" in q):
            if "xgb_features_df" in globals() and xgb_features_df is not None and not xgb_features_df.empty:
                df = xgb_features_df.copy()
                df.columns = [str(c).strip() for c in df.columns]
                score_col = df.columns[1]
                df[score_col] = pd.to_numeric(df[score_col], errors="coerce")
                df = df.sort_values(score_col, ascending=False)

                n = 5 if ("top 5" in q or "five" in q) else min(len(df), 10)
                tool_result = {
                    "analysis_type": "xgboost_feature_importance",
                    "table_used": "xgb_features_df",
                    "result": df.head(n).to_dict(orient="records")
                }
            else:
                tool_result = {"error": "xgb_features_df is not loaded."}

        # 2) Top sentiment countries
        elif "top" in q and "sentiment" in q:
            if not country_summary_df.empty and "avg_sentiment" in country_summary_df.columns:
                df = country_summary_df.copy()
                df["avg_sentiment"] = pd.to_numeric(df["avg_sentiment"], errors="coerce")
                if "positive_share" in df.columns:
                    df["positive_share"] = pd.to_numeric(df["positive_share"], errors="coerce")

                df = df.sort_values("avg_sentiment", ascending=False).head(5)

                keep_cols = [c for c in ["country", "avg_sentiment", "positive_share"] if c in df.columns]
                tool_result = {
                    "analysis_type": "top_sentiment",
                    "table_used": "country_summary",
                    "result": df[keep_cols].to_dict(orient="records")
                }
            else:
                tool_result = {"error": "country_summary data is unavailable or missing avg_sentiment."}

        # 3) Compare countries
        elif "compare" in q and len(country_filters) >= 2:
            if not country_summary_df.empty:
                df = country_summary_df.copy()
                if "country" in df.columns:
                    df = df[df["country"].isin(country_filters)].sort_values("country")
                    tool_result = {
                        "analysis_type": "country_comparison",
                        "table_used": "country_summary",
                        "result": df.to_dict(orient="records")
                    }
                else:
                    tool_result = {"error": "country_summary missing country column."}
            else:
                tool_result = {"error": "No country summary data available for comparison."}

        # 4) Show db_merged rows
        elif "row" in q or "db_merged" in q:
            tool_result = {
                "analysis_type": "db_merged_lookup",
                "table_used": "db_merged_rows",
                "result": db_merged_rows_data
            }

        # 5) NEW: Canada gap / underperform / peers
        elif "underperform" in q or "gap" in q or "peer" in q:
            tool_result = {
                "analysis_type": "gap_analysis",
                "table_used": "db_merged_rows / country_summary",
                "result": {
                    "country_filters_used": country_filters,
                    "country_summary": country_summary_data,
                    "db_merged_rows": db_merged_rows_data,
                    "note": "Use these rows to identify where Canada lags peer countries."
                }
            }

        # 6) NEW: projection / impact / assumptions
        elif "project" in q or "projection" in q or "impact" in q or "assumption" in q:
            tool_result = {
                "analysis_type": "policy_projection",
                "table_used": "db_merged_rows / country_summary",
                "result": {
                    "baseline_country_filters": country_filters,
                    "baseline_country_summary": country_summary_data,
                    "baseline_db_merged_rows": db_merged_rows_data,
                    "assumptions_example": {
                        "ai_usage_percent_increase": "+5 points",
                        "avg_sentiment_increase": "+0.10",
                        "broadband_speed_increase": "+20 mbps"
                    },
                    "note": "This is a scenario scaffold, not a retrained predictive model."
                }
            }

        else:
            tool_result = {
                "analysis_type": "none",
                "table_used": None,
                "result": None
            }

    except Exception as e:
        tool_result = {"error": str(e)}

    return {
        "structured_context": structured,
        "tool_result": tool_result
    }


def writer_agent(
    user_query: str,
    plan,
    research_output,
    analysis_output,
    chat_history,
    critique_output=None,
    force_workflow_display=False
):
    structured_context = analysis_output.get("structured_context", {})
    slim_structured_context = {
        "country_summary": structured_context.get("country_summary", [])[:5],
        "source_country_summary": structured_context.get("source_country_summary", [])[:5],
        "db_merged_rows": structured_context.get("db_merged_rows", [])[:2],
        "xgb_features": structured_context.get("xgb_features", [])[:10],
    }

    tool_context = research_output.get("tool_context", "")
    tool_context_str = json.dumps(tool_context, ensure_ascii=False, default=str) if tool_context else ""

    extra_instruction = ""

    if critique_output:
        extra_instruction += """
You are revising a previous draft using a critique.
Improve the answer by addressing:
- weaknesses
- risks
- missing stakeholder perspectives
- unrealistic assumptions

Return only the improved final answer.
"""

    if force_workflow_display or plan.get("show_workflow", False):
        extra_instruction += """
Structure the answer with these headings exactly when relevant:
[Planner]
[Research Agent]
[Analyst Agent]
[Critic Agent]
[Final Answer]

Under each heading, briefly show what that component contributed.
"""

    prompt = f"""
{WRITER_SYSTEM}

{extra_instruction}

Conversation history:
{json.dumps(chat_history[-2:], indent=2, ensure_ascii=False)}

User query:
{user_query}

Planner output:
{json.dumps(plan, indent=2, ensure_ascii=False)}

Research context:
{research_output.get("research_context", "")[:4000]}

Tool context:
{tool_context_str[:2000]}

Structured context:
{json.dumps(slim_structured_context, indent=2, ensure_ascii=False, default=str)}

Tool result:
{json.dumps(analysis_output.get("tool_result", None), indent=2, ensure_ascii=False, default=str)[:3000]}

Critique:
{critique_output[:2500] if critique_output else ""}
"""
    return call_llm(prompt)
CRITIC_SYSTEM = """
You are a policy critic agent.

Your task:
1. Identify weaknesses in the draft answer
2. Point out implementation risks, missing stakeholders, or weak assumptions
3. Suggest what should be improved

Be concise, specific, and evidence-grounded.
"""

def critic_agent(user_query: str, draft_answer: str, research_output, analysis_output):
    prompt = f"""
{CRITIC_SYSTEM}

User query:
{user_query}

Draft answer:
{draft_answer}

Research context:
{research_output.get("research_context", "")[:3000]}

Tool context:
{research_output.get("tool_context", "")[:1500]}

Tool result:
{json.dumps(analysis_output.get("tool_result", None), indent=2, ensure_ascii=False, default=str)[:2000]}
"""
    return call_llm(prompt)

chat_history = []

def chatbot_answer(user_query: str, return_debug: bool = False):
    plan = planner_agent(user_query)

    # 1) Research
    research_output = {
        "research_context": "Retrieval not used.",
        "tool_context": "",
        "hits": []
    }
    if plan.get("needs_retrieval", False):
        research_output = researcher_agent(user_query, plan.get("country_filters", []))

    # 2) Analysis
    analysis_output = {
        "structured_context": {},
        "tool_result": None
    }
    if (
        plan.get("needs_structured_data", False)
        or plan.get("needs_python_tool", False)
        or plan.get("needs_projection", False)
    ):
        analysis_output = analyst_agent(user_query, plan.get("country_filters", []))

    # 3) First draft
    draft_answer = writer_agent(
        user_query=user_query,
        plan=plan,
        research_output=research_output,
        analysis_output=analysis_output,
        chat_history=chat_history,
        critique_output=None,
        force_workflow_display=plan.get("show_workflow", False)
    )

    # 4) Critique + revise if needed
    critique_output = None
    final_answer = draft_answer

    if plan.get("needs_critique", False):
        critique_output = critic_agent(
            user_query=user_query,
            draft_answer=draft_answer,
            research_output=research_output,
            analysis_output=analysis_output
        )

        final_answer = writer_agent(
            user_query=user_query,
            plan=plan,
            research_output=research_output,
            analysis_output=analysis_output,
            chat_history=chat_history,
            critique_output=critique_output,
            force_workflow_display=plan.get("show_workflow", False)
        )

    # 5) Save history
    chat_history.append({"role": "user", "content": user_query})
    chat_history.append({"role": "assistant", "content": final_answer})

    # 6) Optional debug view for demo / grading
    if return_debug:
        return {
            "plan": plan,
            "research_output": research_output,
            "analysis_output": analysis_output,
            "draft_answer": draft_answer,
            "critique_output": critique_output,
            "final_answer": final_answer
        }

    return final_answer

print("Backend rebuilt successfully.")

Backend rebuilt successfully.


In [ ]:
import gradio as gr
import traceback

EXAMPLES = [
    "Which countries have the most positive AI sentiment overall?",
    "Compare Canada and the United States using both sentiment summaries and document evidence.",
    "What are the main weaknesses in Canada's AI competitiveness?",
    "What are your three main recommendations for improving Canada's AI strategy?",
    "Show me the row for Canada in db_merged_df.",
    "How do you measure AI competitiveness in this project?"
]

assert "chatbot_answer" in globals(), "chatbot_answer is not defined. Run the backend cells first."

def respond(message, chat_history):
    if chat_history is None:
        chat_history = []

    message = (message or "").strip()
    if not message:
        return "", chat_history

    try:
        answer = chatbot_answer(message)
    except Exception:
        err = traceback.format_exc()
        print(err)
        answer = f"Backend error:\n{err}"

    chat_history = chat_history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": answer}
    ]
    return "", chat_history

with gr.Blocks(title="Canada AI Strategy Chatbot") as demo:
    gr.Markdown("# Canada AI Strategy Chatbot")
    gr.Markdown("Ask about Canada's AI strategy, rankings, methodology, or recommendations.")

    chatbot = gr.Chatbot(height=500, type="messages")

    with gr.Row():
        msg = gr.Textbox(
            placeholder="Type your question here...",
            lines=2,
            scale=8
        )
        send = gr.Button("Send", variant="primary", scale=1)

    clear = gr.Button("Clear Chat")

    gr.Markdown("### Example questions")
    example_buttons = []
    with gr.Row():
        for ex in EXAMPLES[:2]:
            btn = gr.Button(ex, size="sm")
            example_buttons.append((btn, ex))
    with gr.Row():
        for ex in EXAMPLES[2:4]:
            btn = gr.Button(ex, size="sm")
            example_buttons.append((btn, ex))
    with gr.Row():
        for ex in EXAMPLES[4:]:
            btn = gr.Button(ex, size="sm")
            example_buttons.append((btn, ex))

    send.click(
        respond,
        inputs=[msg, chatbot],
        outputs=[msg, chatbot]
    )

    msg.submit(
        respond,
        inputs=[msg, chatbot],
        outputs=[msg, chatbot]
    )

    clear.click(lambda: ("", []), outputs=[msg, chatbot])

    for btn, ex in example_buttons:
        btn.click(
            fn=lambda e=ex: e,
            inputs=None,
            outputs=[msg]
        )

demo.queue()
demo.launch(
    share=True,
    inline=False,
    debug=True,
    prevent_thread_lock=True
)

/tmp/ipykernel_21179/3028855191.py:40: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(height=500, type="messages")


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://25c34935be59e51ce8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
hits = retrieve_context(
    "How do you measure AI competitiveness in this project?",
    top_k=10,
    country_filter=None
)

for i, h in enumerate(hits, 1):
    print(h["source"], h["score"])